In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# SkillGap: AI Career Skills Analyzer
# No API needed - pure Python agent

def skill_gap_agent(user_skills: list, target_role: str) -> dict:
    """
    Agent that analyzes skill gaps for target career roles.
    Returns structured report with score, gaps, and roadmap.
    """
    
    # Knowledge base - predefined role requirements
    role_requirements = {
        "data scientist": ["python", "sql", "machine learning", "statistics", 
                          "pandas", "numpy", "visualization", "deep learning"],
        "web developer": ["html", "css", "javascript", "react", "nodejs", 
                         "git", "api", "database"],
        "ai engineer": ["python", "machine learning", "deep learning", "llm",
                       "api", "docker", "cloud", "pytorch"],
        "software engineer": ["python", "data structures", "algorithms", 
                             "git", "sql", "system design", "api", "testing"],
        "data analyst": ["sql", "excel", "python", "visualization", 
                        "statistics", "pandas", "reporting", "dashboard"]
    }
    
    # Normalize inputs
    user_skills_lower = [s.lower().strip() for s in user_skills]
    role_lower = target_role.lower().strip()
    
    # Find closest matching role
    matched_role = None
    for role in role_requirements:
        if role in role_lower or role_lower in role:
            matched_role = role
            break
    
    if not matched_role:
        matched_role = "software engineer"  # default
    
    required_skills = role_requirements[matched_role]
    
    # Calculate match
    matched = [s for s in required_skills if s in user_skills_lower]
    missing = [s for s in required_skills if s not in user_skills_lower]
    score = round((len(matched) / len(required_skills)) * 100)
    
    # Generate learning roadmap
    roadmap = {
        "immediate": missing[:2],   # Learn these first
        "short_term": missing[2:4], # Next 3 months
        "long_term": missing[4:]    # 6+ months
    }
    
    # Security: sanitize inputs before output
    safe_skills = [s[:50] for s in user_skills_lower]  # Limit length
    
    return {
        "target_role": matched_role.title(),
        "your_skills": safe_skills,
        "match_score": score,
        "matched_skills": matched,
        "missing_skills": missing,
        "learning_roadmap": roadmap,
        "verdict": "Strong candidate" if score >= 70 else 
                   "Needs development" if score >= 40 else 
                   "Significant gaps",
        "next_step": f"Start learning {missing[0]} this week" if missing else 
                     "You are ready to apply!"
    }

# ─── TEST THE AGENT ───────────────────────────────────────────────────────────
print("=" * 50)
print("  SkillGap — AI Career Skills Analyzer")
print("=" * 50)

# Example: Student wanting to become AI Engineer
my_skills = ["python", "machine learning", "sql", "git"]
target = "AI Engineer"

result = skill_gap_agent(my_skills, target)

print(f"\n🎯 Target Role: {result['target_role']}")
print(f"📊 Match Score: {result['match_score']}/100")
print(f"✅ Verdict: {result['verdict']}")
print(f"\n✅ Skills You Have: {', '.join(result['matched_skills'])}")
print(f"❌ Skills You're Missing: {', '.join(result['missing_skills'])}")
print(f"\n📚 Learning Roadmap:")
print(f"  This week: {result['learning_roadmap']['immediate']}")
print(f"  3 months:  {result['learning_roadmap']['short_term']}")
print(f"  6 months:  {result['learning_roadmap']['long_term']}")
print(f"\n🚀 Next Step: {result['next_step']}")

In [ ]:
# Cell 2: Interactive Agent - User inputs their own skills
print("=" * 50)
print("  SkillGap — AI Career Skills Analyzer")
print("=" * 50)

# Get user input
print("\nAvailable roles: Data Scientist, Web Developer, AI Engineer, Software Engineer, Data Analyst")
target_role = input("\nEnter your target role: ").strip()
skills_input = input("Enter your current skills (comma separated): ").strip()

# Parse skills
user_skills = [s.strip() for s in skills_input.split(",") if s.strip()]

# Run agent
result = skill_gap_agent(user_skills, target_role)

# Display results
print("\n" + "=" * 50)
print("  YOUR SKILL GAP REPORT")
print("=" * 50)
print(f"\n🎯 Target Role:  {result['target_role']}")
print(f"📊 Match Score:  {result['match_score']}/100")
print(f"✅ Verdict:      {result['verdict']}")
print(f"\n✅ You Have:     {', '.join(result['matched_skills']) or 'None matched'}")
print(f"❌ You're Missing: {', '.join(result['missing_skills'])}")
print(f"\n📚 Learning Roadmap:")
print(f"   This week → {', '.join(result['learning_roadmap']['immediate'])}")
print(f"   3 months  → {', '.join(result['learning_roadmap']['short_term'])}")
print(f"   6 months  → {', '.join(result['learning_roadmap']['long_term'])}")
print(f"\n🚀 Next Step: {result['next_step']}")

# Save report
import json
from datetime import datetime
report = {
    "timestamp": datetime.now().isoformat(),
    "target_role": result['target_role'],
    "match_score": result['match_score'],
    "missing_skills": result['missing_skills'],
    "roadmap": result['learning_roadmap'],
    "next_step": result['next_step']
}
path = f"/kaggle/working/skillgap_report_{datetime.now().strftime('%H%M%S')}.json"
with open(path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"\n📁 Report saved: {path}")

In [ ]:
# Cell 3: Multi-Agent System
# Agent 1: Skill Analyzer | Agent 2: Resource Recommender | Agent 3: Safety Checker

def resource_recommender_agent(missing_skills: list, target_role: str) -> dict:
    """
    Agent 2: Recommends free learning resources for missing skills.
    Focuses on Indian students - free resources only.
    """
    free_resources = {
        "python": "freeCodeCamp Python Course (YouTube) — free",
        "sql": "SQLZoo.net — free interactive SQL practice",
        "machine learning": "Andrew Ng ML Course on Coursera — audit free",
        "deep learning": "fast.ai Practical Deep Learning — completely free",
        "excel": "GFG Excel Tutorial — free",
        "statistics": "Khan Academy Statistics — completely free",
        "pandas": "Kaggle Pandas Course — free",
        "visualization": "Matplotlib & Seaborn on Kaggle — free",
        "javascript": "The Odin Project — completely free",
        "react": "React official docs + Scrimba — free tier",
        "docker": "Docker official tutorial — free",
        "api": "RapidAPI Learn — free",
        "cloud": "Google Cloud Skills Boost — free tier",
        "pytorch": "PyTorch official tutorials — free",
        "llm": "Kaggle LLM course — free",
        "git": "GitHub Skills — completely free",
        "nodejs": "The Odin Project NodeJS — free",
        "algorithms": "NeetCode.io — free DSA practice",
        "data structures": "NeetCode.io — free DSA practice",
        "reporting": "Google Data Studio tutorials — free",
        "dashboard": "Tableau Public — free",
        "numpy": "NumPy official quickstart — free",
        "testing": "pytest official docs — free",
        "system design": "System Design Primer GitHub — free",
    }
    
    recommendations = {}
    for skill in missing_skills[:5]:  # Top 5 missing skills
        skill_lower = skill.lower()
        recommendations[skill] = free_resources.get(
            skill_lower, 
            f"Search '{skill} tutorial' on YouTube or Coursera free audit"
        )
    
    return {
        "role": target_role,
        "resources": recommendations,
        "estimated_time": f"{len(missing_skills) * 3} weeks at 2 hours/day",
        "platform_tip": "All resources above are FREE — no payment needed"
    }

def safety_checker_agent(report: dict) -> dict:
    """
    Agent 3: Security and safety validation.
    Ensures output is safe, appropriate, and doesn't contain harmful advice.
    """
    # Check for harmful content patterns
    harmful_patterns = ["quit", "dropout", "give up", "impossible", "never"]
    
    # Validate score is realistic
    if report.get("match_score", 0) < 10:
        report["encouragement"] = "Everyone starts somewhere. Even 1 matching skill is a foundation to build on!"
    
    # Add safety disclaimer
    report["disclaimer"] = "This is an AI-generated analysis for guidance only. Consult career counselors for personalized advice."
    report["security_verified"] = True
    
    return report

# ─── RUN FULL MULTI-AGENT PIPELINE ───────────────────────────────────────────
print("=" * 55)
print("  SkillGap Multi-Agent Pipeline")
print("=" * 55)

print("\nAvailable roles: Data Scientist, Web Developer, AI Engineer, Software Engineer, Data Analyst")
target_role = input("\nEnter your target role: ").strip()
skills_input = input("Enter your current skills (comma separated): ").strip()
user_skills = [s.strip() for s in skills_input.split(",") if s.strip()]

# Agent 1: Analyze skills
print("\n🤖 Agent 1 (Skill Analyzer): Analyzing your profile...")
result = skill_gap_agent(user_skills, target_role)

# Agent 2: Recommend resources
print("🤖 Agent 2 (Resource Recommender): Finding free learning resources...")
resources = resource_recommender_agent(result["missing_skills"], result["target_role"])

# Agent 3: Safety check
print("🤖 Agent 3 (Safety Checker): Validating output...")
result = safety_checker_agent(result)

# Display full report
print("\n" + "=" * 55)
print("  YOUR COMPLETE CAREER DEVELOPMENT REPORT")
print("=" * 55)
print(f"\n🎯 Target Role:   {result['target_role']}")
print(f"📊 Match Score:   {result['match_score']}/100")
print(f"✅ Verdict:       {result['verdict']}")

if "encouragement" in result:
    print(f"💪 Note:          {result['encouragement']}")

print(f"\n✅ Skills You Have:    {', '.join(result['matched_skills']) or 'Building from scratch!'}")
print(f"❌ Skills to Learn:    {', '.join(result['missing_skills'])}")

print(f"\n📚 Learning Roadmap:")
print(f"   This week → {', '.join(result['learning_roadmap']['immediate'])}")
print(f"   3 months  → {', '.join(result['learning_roadmap']['short_term'])}")
print(f"   6 months  → {', '.join(result['learning_roadmap']['long_term'])}")

print(f"\n🔗 Free Resources for Your Top Missing Skills:")
for skill, resource in resources["resources"].items():
    print(f"   • {skill.title()}: {resource}")

print(f"\n⏱️  Estimated Time: {resources['estimated_time']}")
print(f"💡 Tip: {resources['platform_tip']}")
print(f"\n🚀 Next Step: {result['next_step']}")
print(f"\n⚕️  {result['disclaimer']}")
print(f"🔒 Security verified: {result['security_verified']}")

# Save complete report
import json
from datetime import datetime
full_report = {
    "timestamp": datetime.now().isoformat(),
    "agents_used": ["Skill Analyzer", "Resource Recommender", "Safety Checker"],
    "target_role": result['target_role'],
    "match_score": result['match_score'],
    "matched_skills": result['matched_skills'],
    "missing_skills": result['missing_skills'],
    "roadmap": result['learning_roadmap'],
    "resources": resources['resources'],
    "next_step": result['next_step'],
    "security_verified": result['security_verified'],
    "disclaimer": result['disclaimer']
}
path = f"/kaggle/working/skillgap_full_report_{datetime.now().strftime('%H%M%S')}.json"
with open(path, 'w') as f:
    json.dump(full_report, f, indent=2)
print(f"\n📁 Full report saved: {path}")

In [ ]:
# Cell 4: Wrap as real Google ADK Agent
!pip install google-adk -q

import os, asyncio, time, nest_asyncio
nest_asyncio.apply()

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# ─── YOUR PYTHON FUNCTIONS BECOME ADK TOOLS ───────────────────────────────────
# These are the exact same functions from Cell 1 and Cell 3
# ADK automatically converts them into callable tools for the agent

def analyze_skill_gap(target_role: str, user_skills: str) -> dict:
    """
    Analyzes skill gaps between user's current skills and target role requirements.
    
    Args:
        target_role: The job role the user wants (e.g. 'Data Scientist')
        user_skills: Comma-separated list of user's current skills
    
    Returns:
        Structured skill gap report with score, matched skills, and missing skills
    """
    role_requirements = {
        "data scientist": ["python", "sql", "machine learning", "statistics",
                          "pandas", "numpy", "visualization", "deep learning"],
        "web developer": ["html", "css", "javascript", "react", "nodejs",
                         "git", "api", "database"],
        "ai engineer": ["python", "machine learning", "deep learning", "llm",
                       "api", "docker", "cloud", "pytorch"],
        "software engineer": ["python", "data structures", "algorithms",
                             "git", "sql", "system design", "api", "testing"],
        "data analyst": ["sql", "excel", "python", "visualization",
                        "statistics", "pandas", "reporting", "dashboard"]
        "marketing analyst": ["excel", "sql", "google analytics", "visualization", 
                      "reporting", "python", "dashboard", "statistics"]
    }
    
    skills_list = [s.lower().strip() for s in user_skills.split(",")]
    role_lower = target_role.lower().strip()
    
    matched_role = "software engineer"
    for role in role_requirements:
        if role in role_lower or role_lower in role:
            matched_role = role
            break
    
    required = role_requirements[matched_role]
    matched = [s for s in required if s in skills_list]
    missing = [s for s in required if s not in skills_list]
    score = round((len(matched) / len(required)) * 100)
    
    return {
        "target_role": matched_role.title(),
        "match_score": score,
        "matched_skills": matched,
        "missing_skills": missing,
        "verdict": "Strong candidate" if score >= 70 else
                   "Needs development" if score >= 40 else
                   "Significant gaps",
        "next_step": f"Start learning {missing[0]} this week" if missing else
                     "You are ready to apply!"
    }

def recommend_resources(missing_skills: str) -> dict:
    """
    Recommends free learning resources for missing skills.
    Focuses on free resources accessible to Indian students.
    
    Args:
        missing_skills: Comma-separated list of skills to learn
    
    Returns:
        Dictionary of skills mapped to free learning resources
    """
    free_resources = {
        "python": "freeCodeCamp Python Course on YouTube — free",
        "sql": "SQLZoo.net — free interactive SQL",
        "machine learning": "Andrew Ng ML Course — audit free on Coursera",
        "deep learning": "fast.ai Practical Deep Learning — free",
        "excel": "GFG Excel Tutorial — free",
        "statistics": "Khan Academy Statistics — free",
        "pandas": "Kaggle Pandas Course — free",
        "visualization": "Seaborn & Matplotlib on Kaggle — free",
        "javascript": "The Odin Project — free",
        "react": "React official docs + Scrimba free tier",
        "docker": "Docker official tutorial — free",
        "api": "RapidAPI Learn — free",
        "cloud": "Google Cloud Skills Boost — free tier",
        "pytorch": "PyTorch official tutorials — free",
        "llm": "Kaggle LLM course — free",
        "git": "GitHub Skills — free",
        "algorithms": "NeetCode.io — free",
        "data structures": "NeetCode.io — free",
        "reporting": "Google Looker Studio tutorials — free",
        "dashboard": "Tableau Public — free",
        "numpy": "NumPy official quickstart — free",
    }
    
    skills_list = [s.strip().lower() for s in missing_skills.split(",")]
    recommendations = {}
    for skill in skills_list[:5]:
        recommendations[skill] = free_resources.get(
            skill,
            f"Search '{skill} tutorial' on YouTube — many free options"
        )
    
    return {
        "resources": recommendations,
        "tip": "All resources above are FREE — no payment needed",
        "estimated_time": f"{len(skills_list) * 3} weeks at 2 hours/day"
    }

# ─── BUILD THE ADK AGENT ──────────────────────────────────────────────────────
root_agent = LlmAgent(
    name="SkillGap",
    model="gemini-2.5-flash",
    description="An AI career coach that analyzes skill gaps and recommends free learning resources for students.",
    tools=[analyze_skill_gap, recommend_resources],
    instruction="""
    You are SkillGap — an AI career coach for students and fresh graduates in India.
    You help people understand what skills they need for their target role and how to get them for free.

    When a user tells you their target role and current skills:

    STEP 1: Call analyze_skill_gap(target_role, user_skills) to get their gap report.

    STEP 2: Call recommend_resources(missing_skills) where missing_skills is a
    comma-separated string of the missing skills from Step 1.

    STEP 3: Present the results in this exact format:

    ## 🎯 Your Career Profile
    Role: [target role]
    Match Score: [score]/100
    Verdict: [verdict]

    ## ✅ Skills You Already Have
    [list matched skills]

    ## ❌ Skills You Need
    [list missing skills]

    ## 📚 Free Learning Resources
    [list each skill with its free resource]

    ## ⏱️ Time Estimate
    [estimated time]

    ## 🚀 Your Next Step
    [next step]

    ## ⚕️ Disclaimer
    This is AI-generated guidance. Consult a career counselor for personalized advice.

    Be encouraging, direct, and specific. Always mention that all resources are free.
    """,
)

print("✅ SkillGap ADK agent built")
print(f"✅ Tools: analyze_skill_gap, recommend_resources")
print(f"✅ Model: gemini-2.5-flash")

In [ ]:
# Cell 5: Run SkillGap Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()
APP_NAME, USER_ID, SESSION_ID = "SkillGap", "user_001", "session_001"

runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

async def setup_session():
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)

async def chat(msg):
    content = types.Content(role="user", parts=[types.Part(text=msg)])
    response_text = ""
    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                response_text = event.content.parts[0].text
    return response_text

async def chat_with_retry(msg, max_retries=4):
    for attempt in range(max_retries):
        try:
            return await chat(msg)
        except Exception as e:
            if "429" in str(e) or "503" in str(e):
                wait = (attempt + 1) * 15
                print(f"Waiting {wait}s...")
                time.sleep(wait)
            else:
                raise e
    return "Please retry."

async def run():
    await setup_session()
    print("=" * 55)
    print("  SkillGap — AI Career Coach for Students")
    print("=" * 55)
    opening = await chat_with_retry(
        "Hello! I need help understanding what skills I need for my career."
    )
    print(f"\n🤖 SkillGap: {opening}\n")
    while True:
        user_input = input("👤 You: ").strip()
        if not user_input: continue
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("🤖 SkillGap: Good luck with your career journey! 🚀")
            break
        response = await chat_with_retry(user_input)
        print(f"\n🤖 SkillGap: {response}\n")

asyncio.get_event_loop().run_until_complete(run())